In [1]:
import os
import sys
from pyspark.sql import functions as F

# 1. Point to your specific Anaconda environment's python
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

# 2. Add these to help Java find the right paths on Windows
# Replace with your actual Java path if different, 
# but often simply setting the IPv4 stack helps the handshake
os.environ['_JAVA_OPTIONS'] = "-Djava.net.preferIPv4Stack=true"

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType
from pyspark.ml.feature import FeatureHasher

col_names_molecule_protein = ["ChemicalName", "ChemicalID", "CasRN", "GeneSymbol", "GeneID", "GeneForms", "Organism", "OrganismID", "Interaction", "InteractionActions", "PubMedIDs"]

dataset_path = "data/CTD_chem_gene_ixns/CTD_chem_gene_ixns.csv"

# Create a schema where every column in your array is a StringType
schema = StructType([StructField(name, StringType(), True) for name in col_names_molecule_protein])

# 1. Initialize Spark Session
spark = SparkSession.builder \
    .master("local[*]") \
    .appName("IneractionAnalysis") \
    .config("spark.driver.memory", "4g") \
    .config("spark.ui.showConsoleProgress", "true") \
    .getOrCreate()

print(f"Spark Version: {spark.version}")

# 2. Read the CSV into a Spark DataFrame
# Replace 'your_data.csv' with your actual file path
df = spark.read.csv(dataset_path, header=True, schema=schema)

df.show()

Spark Version: 2.4.5
+--------------------+----------+-----+----------+------+---------+------------+----------+--------------------+--------------------+-----------------+
|        ChemicalName|ChemicalID|CasRN|GeneSymbol|GeneID|GeneForms|    Organism|OrganismID|         Interaction|  InteractionActions|        PubMedIDs|
+--------------------+----------+-----+----------+------+---------+------------+----------+--------------------+--------------------+-----------------+
|            10074-G5|   C534883| null|        AR|   367|  protein|Homo sapiens|      9606|10074-G5 inhibits...|decreases^reactio...|         32184358|
|            10074-G5|   C534883| null|        AR|   367|  protein|Homo sapiens|      9606|10074-G5 results ...|decreases^expression|         32184358|
|            10074-G5|   C534883| null|        AR|   367|  protein|Homo sapiens|      9606|10074-G5 results ...|decreases^expression|         32184358|
|            10074-G5|   C534883| null|     EPHB2|  2048|  protein|

In [3]:

#df_interaction = df.filter(F.col("GeneForms") == "protein").filter(F.col("Organism") == "Homo sapiens").select("Interaction").distinct()
#df_interaction = df.filter(F.col("GeneForms") == "protein").select("Interaction").distinct()
df_interaction = df.select("Interaction").distinct()
print(df_interaction.count())

2195511


In [4]:
#df_interaction.show(truncate=False)

In [5]:

#"results in", 
# 1. Your list of phrases (easy to add more here)
 actions = ["results in decreased activity of", "results in increased metabolism of",
           "results in increased expression of", "results in decreased expression of",
           "results in increased chemical synthesis of", "results in increased cleavage of",
           "results in increased methylation of", "results in increased glucuronidation of",
           "results in increased susceptibility to", "results in increased activity of",
            "results in increased phosphorylation of", 
           "results in increased", "results in decreased methylation of",
           "results in decreased susceptibility to", "results in decreased phosphorylation of",   
           "results in decreased metabolism of", "results in decreased oxidation of",
           "results in decreased glycosylation of", "results in decreased secretion of",
           "results in decreased acetylation of", "results in decreased chemical synthesis of",
           "results in decreased sumoylation of", "results in decreased",
           "results in decreased cleavage of", "results in decreased degradation of",
           "inhibits the reaction", "catalyzed by", "co-treated with", "Caproate binds to", 
           "binds to", "which binds to", "affects the expression of", "promotes the reaction",
          "affects the activity of", "affects the phosphorylation of", "affects the localization of",
          "affects the susceptibility to", "affects the uptake of", "affects the transport of",
          "affects the metabolism of", "affects the oxidation of", "affects the reaction",
          "affects the secretion of", "affects the sumoylation of", "affects the chemical synthesis of",
          "affects the acetylation of", "affects the cleavage of", "affects the farnesylation of",
          "affects the import of", "form affects the glucuronidation of", "affects the export of",
          "affects the abundance of", "affects the folding of", "affects the folding of", "affects the reduction",
          "affects the glucuronidation of", "affects the stability of", "affects the methylation of",
          "affects the sulfation of", "affects the stability of", "affects the sulfation of",
          "affects the hydroxylation of", "affects the degradation of", "affects the glycosylation of",
          "affects the hydrolysis of", "affects the lipidation of", "affects the hydrolysis of",
          "affects the alkylation of", "affects the ubiquitination of", "affects the glutathionylation of",
          "affects the carboxylation of", "affects the glutathionylation of", "affects the glycation of",
          "affects the splicing of",
          "affects the mutagenesis of"]



pattern = "|".join(actions)

# 2. Build a list of 'when' expressions dynamically
df_filtered = df_interaction.withColumn("Found_Interaction", F.col("Interaction").rlike(pattern))

#df_filtered.show()

In [6]:
df_filtered.filter(F.col("Found_Interaction") == False).show(truncate=False)

+-----------+-----------------+
|Interaction|Found_Interaction|
+-----------+-----------------+
+-----------+-----------------+

